# Case study: exploring the EBTI decisions

Session 5 · Exploratory analysis and statistics recap. Author: course team, licence CC-BY-4.0.

The case study: European Binding Tariff Information (BTI) decisions. A trader describes a product, a customs authority states its code in the nomenclature, and the European Commission publishes the decision in the EBTI database. The course task (from Session 8) is to predict the four-digit HS heading from the description. This notebook explores the data first and follows the three practice tasks of the session:

1. **Block 1**: languages, description length and decisions per month; the redesign of a poor chart.
2. **Block 2**: are German descriptions longer than French ones? t-test, Mann–Whitney test, confidence interval and effect sizes; issuing country × section and German language × chapter 85 with chi-square and Cramér's V.
3. **Block 3**: correlation of description length and number of keywords (exploratory only), a check for confounding by language, and a one-page summary or dashboard.

Run it from anywhere inside the repository; the data are read from `case-study/data/` (see `case-study/README.md`). Exercises are marked **Exercise**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats.contingency import association

# find the repository root (the folder that contains case-study/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
decisions = pd.read_parquet(DATA / "train_sample.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet")
monthly_counts = pd.read_parquet(DATA / "monthly_counts.parquet")

decisions["n_chars"] = decisions["description"].str.len()
decisions["n_keywords"] = decisions["keywords"].str.split(",").str.len()
decisions["validity_days"] = (decisions["end_date"] - decisions["start_date"]).dt.days
decisions["year"] = decisions["start_date"].dt.year
decisions = decisions.merge(nomenclature[["heading", "section", "section_name"]], on="heading", how="left")
print(decisions.shape)
decisions[["bti_reference", "issuing_country", "language", "start_date", "heading", "description"]].head(3)

To read a decision you do not understand, look at the English keywords and the English heading description. Keywords and the justification are **not** model inputs later (they are written by customs during or after the classification), but they are fine for exploration.

In [ ]:
example = decisions.iloc[0]
print(example["description"][:400], "...")
print("keywords:", example["keywords"])
print("heading:", example["heading"], "-", nomenclature.set_index("heading").loc[example["heading"], "heading_description"])

## Block 1 · Describe the data and design charts

### 1.1 Variable types and summaries

`issuing_country`, `language`, `status` and `heading` are nominal (the heading is a code, not a quantity); the number of keywords is a count; description length and validity in days are numeric; `start_date` is a date. Summaries should match the type.

In [ ]:
decisions[["n_chars", "n_keywords", "validity_days"]].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).round(1)

In [ ]:
# counts and shares: report both
pd.DataFrame({"n": decisions["language"].value_counts(),
              "share": decisions["language"].value_counts(normalize=True).round(3)}).head(8)

In [ ]:
# the most frequent headings, with their English names
top = (decisions["heading"].value_counts().head(10).rename("n").reset_index()
       .merge(nomenclature[["heading", "heading_description"]], on="heading"))
top["share"] = (top["n"] / len(decisions)).round(3)
top["heading_description"] = top["heading_description"].str[:70]
top

In [ ]:
# the long tail: how many headings have few decisions?
per_heading = decisions["heading"].value_counts()
print("headings in the sample:", per_heading.size)
print("headings with fewer than 10 decisions:", (per_heading < 10).sum())
print("share of decisions in the 50 largest headings:", round(per_heading.head(50).sum() / len(decisions), 3))

### 1.2 Languages, description length and decisions per month

In [ ]:
counts = decisions["language"].value_counts().head(8)
monthly = decisions.set_index("start_date").resample("MS").size()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), layout="constrained")
bars = axes[0].bar(counts.index, counts, color="0.4", width=0.7)
bars[0].set_color("#0072B2")
axes[0].bar_label(bars, labels=[f"{v:.0%}" for v in counts / len(decisions)], padding=2, fontsize=8)
axes[0].set(xlabel="language of the description", ylabel="decisions",
            title="57 % of the decisions are written in German")
axes[0].yaxis.set_major_formatter("{x:,.0f}")

bins = np.logspace(np.log10(decisions["n_chars"].min()), np.log10(decisions["n_chars"].max()), 45)
axes[1].hist(decisions["n_chars"], bins=bins, color="0.5", edgecolor="white")
axes[1].set_xscale("log")
axes[1].axvline(decisions["n_chars"].median(), color="#0072B2", lw=2)
axes[1].set(xlabel="characters per description (log scale)", ylabel="decisions",
            title=f"Median {decisions['n_chars'].median():.0f} characters")

axes[2].plot(monthly.index, monthly, color="0.3")
axes[2].set(ylabel="decisions per month (sample)", ylim=(0, None),
            title="Decisions per month by start of validity")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

**Exercise.** Why does the sample of 50,000 decisions show about 600–900 decisions per month? What would the full training data show (`monthly_counts` has the full counts)? Are there months without data?

### 1.3 Improve a poorly designed chart

The chart below shows the number of decisions per year from `monthly_counts`. It has a truncated axis, rainbow colours, no units and a title that says nothing.

**Exercise.** Redesign it with the four critique questions (question, encoding, accuracy, clarity). Start the bars at zero, use colour only for a distinction that matters (for example the United Kingdom, which stopped issuing decisions after Brexit), and write the finding as the title. Compare with the right panel of `theory/figures/good-vs-poor-chart.png` only after you have tried.

In [ ]:
yearly = (monthly_counts.assign(year=monthly_counts["month"].dt.year)
          .query("2015 <= year <= 2025").groupby("year")["n_decisions"].sum())

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(yearly.index, yearly, color=plt.cm.rainbow(np.linspace(0, 1, len(yearly))))
ax.set_ylim(38000, 52500)
ax.set_title("Decisions");

In [ ]:
# Exercise: your improved chart
# fig, ax = plt.subplots(figsize=(6, 3.5), layout="constrained")
# ...

## Block 2 · Comparing groups

### 2.1 Numeric outcome: are German descriptions longer than French ones?

We compare German and French decisions with Welch's t-test (means) and the Mann–Whitney test (ranks), and report the difference with a 95 % confidence interval, Cohen's d and the common-language effect size (CLES).

In [ ]:
def compare_groups(df: pd.DataFrame, column: str, a_mask: pd.Series, b_mask: pd.Series) -> pd.Series:
    """Welch t-test, Mann-Whitney U, CI and effect sizes for a two-group comparison."""
    a = df.loc[a_mask, column].dropna()
    b = df.loc[b_mask, column].dropna()
    t = stats.ttest_ind(a, b, equal_var=False)
    ci = t.confidence_interval()
    mw = stats.mannwhitneyu(a, b)
    pooled_sd = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
    return pd.Series({
        "n A": len(a), "n B": len(b), "mean A": a.mean(), "mean B": b.mean(),
        "median A": a.median(), "median B": b.median(),
        "difference A - B": a.mean() - b.mean(), "CI low": ci.low, "CI high": ci.high,
        "p (Welch t)": t.pvalue, "p (Mann-Whitney)": mw.pvalue,
        "Cohen's d": (a.mean() - b.mean()) / pooled_sd,
        "CLES P(A > B)": mw.statistic / (len(a) * len(b)),
    })

german, french = decisions["language"].eq("de"), decisions["language"].eq("fr")
compare_groups(decisions, "n_chars", german, french).round(4)

**Reading.** German descriptions are about 465 characters longer (95 % CI about 458 to 472), d ≈ 1.4: a very large difference. A random German description is longer than a random French one about 90 % of the time.

Now the number of keywords, first on the sample, then on the full training set (309,529 decisions).

In [ ]:
compare_groups(decisions, "n_keywords", german, french).round(4)

In [ ]:
full = pd.read_parquet(DATA / "train.parquet", columns=["language", "keywords", "chapter", "issuing_country", "start_date"])
full["n_keywords"] = full["keywords"].str.split(",").str.len()
compare_groups(full, "n_keywords", full["language"].eq("de"), full["language"].eq("fr")).round(4)

The keyword difference is "significant" in both, with a much smaller p-value on the full data, but d stays around 0.1 or below. **The p-value changed because n changed, not because the effect changed.**

**Exercise.** Write two sentences for a customs analyst: what was compared, how large the difference is in characters, and why the language alone cannot tell us whether German administrations write more detailed decisions.

### 2.2 Categorical data: issuing country × section and German language × chapter 85

A contingency table, the chi-square test of independence and Cramér's V. First a large table: do the six largest issuing countries classify the same kinds of goods (sections of the nomenclature)?

In [ ]:
largest = decisions["issuing_country"].value_counts().index[:6]
sub = decisions[decisions["issuing_country"].isin(largest)].dropna(subset=["section"])
table = pd.crosstab(sub["issuing_country"], sub["section"])
res = stats.chi2_contingency(table)
print(f"chi2 = {res.statistic:.0f}, dof = {res.dof}, p = {res.pvalue:.2g}")
print(f"Cramér's V = {association(table, method='cramer'):.3f}")
print("cells with expected count < 5:", (res.expected_freq < 5).sum(), "of", table.size)
# row shares: which sections are over-represented in which country?
(pd.crosstab(sub["issuing_country"], sub["section"], normalize="index") * 100).round(1)

In [ ]:
section_names = nomenclature.dropna(subset=["section_name"]).drop_duplicates("section").set_index("section")["section_name"]
section_names.str[:60]

Second, a 2 × 2 table: is chapter 85 (electrical machinery and equipment) more frequent among German-language decisions?

In [ ]:
g, c = decisions["language"].eq("de").rename("german"), decisions["chapter"].eq("85").rename("chapter_85")
table = pd.crosstab(g, c)
res = stats.chi2_contingency(table)
print(f"chi2 = {res.statistic:.2f}, dof = {res.dof}, p = {res.pvalue:.4f}")
print(f"Cramér's V = {association(table, method='cramer'):.3f}")
expected = pd.DataFrame(res.expected_freq, index=table.index, columns=table.columns).round(0)
display(table, expected)
pd.crosstab(g, c, normalize="index").round(3)

**Significant but negligible.** p is below 0.01, but Cramér's V ≈ 0.014 and the share of chapter 85 differs by about one percentage point (15.1 % vs 14.1 %). The country × section table is different: V ≈ 0.19, a small-to-moderate association.

**Exercise.** Repeat the 2 × 2 test on `full`: how do p and V change? Then name one reason why the country × section association matters for a heading classifier trained on all countries together.

In [ ]:
# Exercise: chi-square on the full training set
# table_full = pd.crosstab(full["language"].eq("de"), full["chapter"].eq("85"))
# ...

## Block 3 · Description length and number of keywords

### 3.1 Scatter plot and correlation

This is an exploratory relationship only: keywords are assigned by customs and are not available when a new description arrives, so they never become a model input. Length is right-skewed, so we also use a log scale, and compare Pearson's r (linear) with Spearman's ρ (monotonic, rank-based).

In [ ]:
k = decisions.dropna(subset=["n_keywords"])
chars, keywords = k["n_chars"], k["n_keywords"]
result = pd.Series({
    "Pearson r (raw)": stats.pearsonr(chars, keywords).statistic,
    "Spearman rho": stats.spearmanr(chars, keywords).statistic,
    "Pearson r (log length)": stats.pearsonr(np.log(chars), keywords).statistic,
})
print(result.round(3))

# sensitivity: drop the five longest descriptions
keep = ~chars.index.isin(chars.nlargest(5).index)
print("Pearson r without the 5 longest:", round(stats.pearsonr(chars[keep], keywords[keep]).statistic, 3))

In [ ]:
sample = k.sample(5000, random_state=1)
rng = np.random.default_rng(0)
lx = np.log(k["n_chars"])
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.scatter(np.log(sample["n_chars"]), sample["n_keywords"] + rng.uniform(-0.3, 0.3, len(sample)),
           s=4, alpha=0.25, color="0.5", label="decision (jittered)")
bins = pd.cut(lx, np.arange(2, 9.5, 0.5))
means = pd.DataFrame({"x": lx, "y": k["n_keywords"]}).groupby(bins, observed=True).mean()
ax.plot(means["x"], means["y"], "o", color="#0072B2", label="mean per length bin")
b1, b0 = np.polyfit(lx, k["n_keywords"], 1)
grid = np.linspace(lx.min(), lx.max(), 2)
ax.plot(grid, b0 + b1 * grid, color="#D55E00", lw=2, label=f"least-squares line, slope {b1:.2f}")
ax.set(xlabel="log(characters in description)", ylabel="number of keywords", ylim=(0, 20),
       title="Longer descriptions come with somewhat more keywords")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### 3.2 Is the language a confounder?

German descriptions are long, French ones short. If the number of keywords depends less on the language than the length does, pooling the languages distorts the relationship. Check it within each language and with the language as a control variable.

In [ ]:
main = k[k["language"].isin(["de", "fr", "en", "nl", "pl"])].assign(log_chars=lambda d: np.log(d["n_chars"]))
within = main.groupby("language").apply(lambda g: pd.Series({
    "n": len(g),
    "median chars": g["n_chars"].median(),
    "median keywords": g["n_keywords"].median(),
    "spearman": stats.spearmanr(g["n_chars"], g["n_keywords"]).statistic,
    "slope (keywords per log char)": np.polyfit(g["log_chars"], g["n_keywords"], 1)[0],
}), include_groups=False)
display(within.round(3))
pooled = smf.ols("n_keywords ~ log_chars", data=main).fit()
adjusted = smf.ols("n_keywords ~ log_chars + C(language)", data=main).fit()
print("pooled slope:", round(pooled.params["log_chars"], 3), " slope with language held fixed:",
      round(adjusted.params["log_chars"], 3))

**Exercise.** Is the relationship stronger or weaker within languages than pooled? Is there a reversal of sign (Simpson's paradox) or only a change in size? We did not find a clear reversal in these data; try one more grouping of your choice (issuing country, section, year) and report honestly what you find.

### 3.3 Language shift over time: a finding for the report

In [ ]:
group = decisions["language"].where(decisions["language"].isin(["de", "fr", "en"]), "other")
share = pd.crosstab(decisions["year"], group, normalize="index")[["de", "fr", "en", "other"]]
gb = monthly_counts[monthly_counts["issuing_country"] == "GB"].groupby(monthly_counts["month"].dt.year)["n_decisions"].sum()
display(share.round(3))
print("decisions issued by the United Kingdom per year:", gb.loc[2016:].to_dict())
# 2055 and 2200 are typing errors in the start dates of the raw export (Session 4)

### 3.4 A one-page summary for a customs analyst

Use the four-part structure for each finding: **finding** (one sentence), **number** with context, **meaning** for the reader, **limitation**.

**Exercise.** Write three findings below (language shift after Brexit, German vs French description length, the long tail of headings), each with one chart from this notebook. Alternatively, extend the Streamlit dashboard of decisions per month by country and chapter in `dashboard_app.py` (start it from the repository root with `uv run streamlit run sessions/05-eda-and-statistics/workbooks/dashboard_app.py`).

*Your summary here.*

1. **Finding:** …  **Number:** …  **Meaning:** …  **Limitation:** …
2. …
3. …